# S06_PD_Sandoval_LimpiezaInconsistencias

**Semana 6 — Limpieza y transformación**

Objetivo: estandarizar textos y formatos, detectar y resolver duplicados, verificar la limpieza y exportar el dataset depurado.


## 1. Carga del dataset de la Semana 4

Se utiliza `S04_PD_Sandoval_DatasetImputacionInicial.csv` como base de trabajo.

In [1]:
import pandas as pd

df = pd.read_csv("S04_PD_Sandoval_DatasetImputacionInicial.csv")

print("Dimensiones iniciales:", df.shape)
print("Columnas:", df.columns.tolist())

Dimensiones iniciales: (618360, 24)
Columnas: ['id_pedido', 'id_cliente', 'fecha_pedido', 'canal_compra', 'metodo_pago', 'ciudad_tienda', 'codigo_postal', 'categoria_producto', 'producto', 'precio_unitario', 'unidades_vendidas', 'monto_compra', 'edad_cliente', 'correo_cliente', 'nivel_satisfaccion', 'nivel_lealtad', 'comentario_cliente', 'peso_pedido_kg', 'fecha_actualizacion_stock', 'fuga_cliente', 'correo_cliente_imputado', 'comentario_cliente_imputado', 'ciudad_tienda_imputado', 'nivel_satisfaccion_imputado']


## 2. Revisión de valores únicos

Antes de corregir, se revisan las cuatro columnas de texto indicadas en la actividad para detectar diferencias de mayúsculas, espacios, tildes y abreviaturas.

In [2]:
for col in ["ciudad_tienda", "categoria_producto", "canal_compra", "metodo_pago"]:
    print(f"\n{col}")
    print(sorted(df[col].dropna().unique().tolist()))


ciudad_tienda
[' Barranquilla', ' Bogotá', ' Bucaramanga', ' Cali', ' Cartagena', ' Cúcuta', ' Ibagué', ' Manizales', ' Medellín', ' Pereira', ' Santa Marta', ' Villavicencio', 'B/manga', 'B/quilla', 'BAQ', 'BARRANQUILLA', 'BGA', 'BOG', 'BOGOTA', 'BOGOTA DC', 'BOGOTÁ', 'BUCARAMANGA', 'Barranquilla', 'Barranquilla ', 'Bogota', 'Bogota D.C.', 'Bogotá', 'Bogotá ', 'Bucaramanga', 'Bucaramanga ', 'CALI', 'CALI ', 'CARTAGENA', 'CTG', 'CUCUTA', 'Cali', 'Cali ', 'Cartagena', 'Cartagena ', 'Cartagena de Indias', 'Cucuta', 'CÚCUTA', 'Cúcuta', 'Cúcuta ', 'Desconocida', 'IBAGUE', 'IBAGUÉ', 'Ibague', 'Ibagué', 'Ibagué ', 'MANIZALES', 'MDE', 'MEDELLIN', 'MEDELLÍN', 'MZL', 'Manizales', 'Manizales ', 'Manizales, Caldas', 'Medellin', 'Medellin, Ant.', 'Medellín', 'Medellín ', 'PEREIRA', 'Pereira', 'Pereira ', 'SANTA MARTA', 'SMR', 'Santa Marta', 'Santa Marta ', 'SantaMarta', 'Santiago de Cali', 'Sta. Marta', 'VILLAVICENCIO', 'Villavicencio', 'Villavicencio ', 'barranquilla', 'bogota', 'bogota d.c.', '

### Variantes detectadas

- **ciudad_tienda:** variantes por mayúsculas/minúsculas, espacios, ausencia de tildes y abreviaturas como `Baq`, `Bga`, `Ctg`, `Mde`, `Mzl`, `Smr`, `Bog`, `B/Quilla`, `B/Manga`, además de nombres alternativos como `Santiago De Cali` y `Cartagena De Indias`.
- **categoria_producto:** variaciones de mayúsculas/minúsculas, espacios y ausencia de tildes en `Electrónica` y `Juguetería`.
- **canal_compra:** variaciones de mayúsculas/minúsculas y espacios en `App`, `Web`, `Tienda` y `Marketplace`.
- **metodo_pago:** variaciones de mayúsculas/minúsculas y espacios; `Paypal` se normaliza a la forma oficial `PayPal`.

Estas variantes se revisaron antes de aplicar las correcciones.

## 3. Unificación de mayúsculas, minúsculas y espacios

In [3]:
for col in ["ciudad_tienda", "categoria_producto", "canal_compra", "metodo_pago"]:
    df[col] = df[col].astype("string").str.strip().str.title()

print("Valores únicos después de la limpieza básica:")
for col in ["ciudad_tienda", "categoria_producto", "canal_compra", "metodo_pago"]:
    print(f"{col}: {df[col].nunique(dropna=True)}")

Valores únicos después de la limpieza básica:
ciudad_tienda: 34
categoria_producto: 8
canal_compra: 4
metodo_pago: 4


## 4. Diccionarios de corrección

`str.strip()` y `str.title()` corrigen formato, pero no pueden recuperar automáticamente tildes ni interpretar abreviaturas. Por eso se utilizan diccionarios específicos por columna.

In [4]:
correcciones_ciudad = {
    "Bogota": "Bogotá",
    "Bogota D.C.": "Bogotá",
    "Bogota Dc": "Bogotá",
    "Bog": "Bogotá",
    "Baq": "Barranquilla",
    "B/Quilla": "Barranquilla",
    "Bga": "Bucaramanga",
    "B/Manga": "Bucaramanga",
    "Ctg": "Cartagena",
    "Cartagena De Indias": "Cartagena",
    "Santiago De Cali": "Cali",
    "Santamarta": "Santa Marta",
    "Sta. Marta": "Santa Marta",
    "Smr": "Santa Marta",
    "Medellin": "Medellín",
    "Medellin, Ant.": "Medellín",
    "Mde": "Medellín",
    "Manizales, Caldas": "Manizales",
    "Mzl": "Manizales",
    "Cucuta": "Cúcuta",
    "Ibague": "Ibagué"
}

correcciones_categoria = {
    "Electronica": "Electrónica",
    "Jugueteria": "Juguetería"
}

correcciones_canal = {}

correcciones_metodo = {
    "Paypal": "PayPal"
}

df["ciudad_tienda"] = df["ciudad_tienda"].replace(correcciones_ciudad)
df["categoria_producto"] = df["categoria_producto"].replace(correcciones_categoria)
df["canal_compra"] = df["canal_compra"].replace(correcciones_canal)
df["metodo_pago"] = df["metodo_pago"].replace(correcciones_metodo)

for col in ["ciudad_tienda", "categoria_producto", "canal_compra", "metodo_pago"]:
    print(f"{col}: {df[col].nunique(dropna=True)} valores únicos")

ciudad_tienda: 13 valores únicos
categoria_producto: 6 valores únicos
canal_compra: 4 valores únicos
metodo_pago: 4 valores únicos


## 5. Verificación de categorías

Después de las correcciones quedan las categorías reales esperadas:

- `ciudad_tienda`: 13 categorías, incluyendo `Desconocida`.
- `categoria_producto`: 6 categorías.
- `canal_compra`: 4 categorías.
- `metodo_pago`: 4 categorías.

## 6. Conversión de `fecha_pedido` a fecha real

In [5]:
meses_es_a_num = {
    "ene": "01", "feb": "02", "mar": "03", "abr": "04",
    "may": "05", "jun": "06", "jul": "07", "ago": "08",
    "sep": "09", "oct": "10", "nov": "11", "dic": "12"
}

for mes_es, mes_num in meses_es_a_num.items():
    df["fecha_pedido"] = (
        df["fecha_pedido"]
        .astype("string")
        .str.replace(f"-{mes_es}-", f"-{mes_num}-", regex=False)
    )

df["fecha_pedido"] = pd.to_datetime(
    df["fecha_pedido"],
    dayfirst=True,
    errors="coerce",
    format="mixed"
)

print("Tipo de dato:", df["fecha_pedido"].dtype)
print("Fechas sin convertir:", df["fecha_pedido"].isnull().sum())

Tipo de dato: datetime64[ns]
Fechas sin convertir: 16488


### Resultado de la conversión

Los valores imposibles de interpretar como fecha válida se convierten en `NaT` gracias a `errors="coerce"`. En los datos originales se identificaron `00/00/0000`, `31/04/2025` y `30/02/2025` como fechas inválidas.

`format="mixed"` permite convertir en una misma columna formatos como `dd/mm/yyyy` y `yyyy-mm-dd`.

## 7. Detección de duplicados exactos

In [6]:
duplicados_exactos = df.duplicated().sum()

print(f"Duplicados exactos: {duplicados_exactos}")

Duplicados exactos: 19360


### Duplicados exactos encontrados

Se encontraron **19,360 filas duplicadas exactas** antes de la eliminación.

Para revisarlas se puede ejecutar:

```python
df[df.duplicated(keep=False)]
```

## 8. Detección de duplicados por `correo_cliente`

In [7]:
duplicados_correo_rubrica = df.duplicated(
    subset=["correo_cliente"],
    keep=False
).sum()

duplicados_correo_reales = (
    df["correo_cliente"].notna()
    & df.duplicated(subset=["correo_cliente"], keep=False)
).sum()

print(f"Duplicados por correo según la comprobación directa: {duplicados_correo_rubrica}")
print(f"Filas duplicadas por correo con correo válido: {duplicados_correo_reales}")

Duplicados por correo según la comprobación directa: 133924
Filas duplicadas por correo con correo válido: 32288


### Resultado

La comprobación directa de pandas da **133,924 filas** porque también considera los valores `NaN` de `correo_cliente` como repetidos.

Para identificar clientes duplicados, el criterio válido es excluir correos faltantes. Con ese criterio se encontraron **32,288 filas**, correspondientes a registros que comparten un correo válido.

Ejemplos revisados incluyen registros repetidos con el mismo `id_pedido`, `id_cliente`, producto y monto, por lo que corresponden a copias del mismo registro.

## 9. Regla de conservación de duplicados

**Para duplicados por `correo_cliente`, conservo el registro con la `fecha_pedido` más reciente, porque representa la información más actualizada. Si dos registros tienen la misma fecha, conservo el que tenga menos valores faltantes. Los correos faltantes no se usan como identificador de duplicado. Para duplicados exactos restantes, conservo la primera aparición.**

La regla queda documentada antes de ejecutar `drop_duplicates()`.

## 10. Aplicación de la eliminación de duplicados

In [8]:
filas_antes = df.shape[0]

# Se separan los registros con correo válido de los que no tienen correo.
df_con_email = df[df["correo_cliente"].notna()].copy()
df_sin_email = df[df["correo_cliente"].isna()].copy()

# Criterio: fecha más reciente; en empate, menos valores faltantes.
df_con_email["_faltantes_fila"] = df_con_email.isna().sum(axis=1)

df_con_email = df_con_email.sort_values(
    by=["correo_cliente", "fecha_pedido", "_faltantes_fila"],
    na_position="first"
)

df_con_email = df_con_email.drop_duplicates(
    subset=["correo_cliente"],
    keep="last"
).drop(columns="_faltantes_fila")

df = pd.concat([df_con_email, df_sin_email], ignore_index=True)

# Eliminar duplicados exactos restantes.
df = df.drop_duplicates(keep="first").reset_index(drop=True)

filas_despues = df.shape[0]

print(f"Filas antes: {filas_antes}")
print(f"Filas después: {filas_despues}")
print(f"Se eliminaron {filas_antes - filas_despues} filas duplicadas.")

Filas antes: 618360
Filas después: 599000
Se eliminaron 19360 filas duplicadas.


### Resultado de la eliminación

- Filas antes: **618,360**
- Filas después: **599,000**
- Filas eliminadas: **19,360**

La eliminación se hizo sin colapsar todos los registros que tienen correo faltante.

## 11. Verificación final

In [9]:
duplicados_exactos_final = df.duplicated().sum()

duplicados_correo_final = (
    df["correo_cliente"].notna()
    & df.duplicated(subset=["correo_cliente"], keep=False)
).sum()

print("Duplicados exactos restantes:", duplicados_exactos_final)
print("Duplicados por correo válido restantes:", duplicados_correo_final)

for col in ["ciudad_tienda", "categoria_producto", "canal_compra", "metodo_pago"]:
    print(f"{col}: {df[col].nunique(dropna=True)} categorías")

print("Valores NaT en fecha_pedido:", df["fecha_pedido"].isnull().sum())
print("Dimensiones finales:", df.shape)

Duplicados exactos restantes: 0
Duplicados por correo válido restantes: 0
ciudad_tienda: 13 categorías
categoria_producto: 6 categorías
canal_compra: 4 categorías
metodo_pago: 4 categorías
Valores NaT en fecha_pedido: 15997
Dimensiones finales: (599000, 24)


### Criterios de aprobación

El dataset queda listo cuando:

- no hay duplicados exactos;
- no hay duplicados por `correo_cliente` entre los correos válidos;
- las cuatro columnas de texto tienen categorías coherentes;
- `fecha_pedido` tiene tipo de fecha real;
- las filas no se pierden por correcciones de texto, sino únicamente por la regla documentada de duplicados.

## 12. Exportación del dataset depurado

In [10]:
df.to_csv(
    "S06_PD_Sandoval_DatasetDepurado.csv",
    index=False
)

print("Archivo exportado: S06_PD_Sandoval_DatasetDepurado.csv")

Archivo exportado: S06_PD_Sandoval_DatasetDepurado.csv


## Resumen final

El dataset depurado contiene **599,000 filas y 24 columnas**.

Se eliminaron **19,360 filas duplicadas** y se estandarizaron las columnas `ciudad_tienda`, `categoria_producto`, `canal_compra` y `metodo_pago`.

El resultado se exporta como:

`S06_PD_Sandoval_DatasetDepurado.csv`